<a href="https://colab.research.google.com/github/dennzii/Low-Memory-Difusion-Based-Virtual-Try-On-via-Quantized-Stable-Difusion-Backbones/blob/main/concat_selfattn_sd1_5_lora_finetune2_04_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import torch
from diffusers import DiffusionPipeline
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("marquis03/high-resolution-viton-zalando-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'high-resolution-viton-zalando-dataset' dataset.
Path to dataset files: /kaggle/input/high-resolution-viton-zalando-dataset


In [ ]:
import os
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as T


class VITONDataset(Dataset):

    def __init__(self, dataset_path, image_size=(512, 512), latent_size=(64, 128)):
        self.dataset_path = dataset_path
        self.image_size = image_size
        self.latent_size = latent_size

        self.file_names = sorted(os.listdir(os.path.join(dataset_path, "image")))

        self.to_tensor = T.ToTensor()

    def __len__(self):
        return len(self.file_names)

    def load_rgb(self, path):
        img = Image.open(path).convert("RGB")
        return img.resize(self.image_size, Image.BILINEAR)

    def concat_images(self, cloth, image):
        w, h = cloth.size
        canvas = Image.new("RGB", (2 * w, h))
        canvas.paste(cloth, (0, 0))
        canvas.paste(image, (w, 0))
        return canvas

    def zero_pad_left(self, pil_img):
        w, h = pil_img.size
        canvas = Image.new(pil_img.mode, (2 * w, h), 0)
        canvas.paste(pil_img, (w, 0))
        return canvas

    def extract_mask_from_agnostic(self, agnostic_img):
        """
        Agnostic görselden üst kıyafet maskesi çıkarır.
        """

        # 1️⃣ Tensor (512x512)
        tensor = self.to_tensor(agnostic_img)

        r, g, b = tensor[0], tensor[1], tensor[2]

        # 2️⃣ Gri alan tespiti (orijinal çözünürlükte)
        mask = (
            (torch.abs(r - g) < 0.02) &
            (torch.abs(r - b) < 0.02) &
            (r > 0.45) & (r < 0.75)
        ).float().unsqueeze(0)  # (1,H,W)

        # 3️⃣ Küçük boşlukları kapat (morphological closing)
        mask = F.max_pool2d(mask.unsqueeze(0), 5, 1, 2).squeeze(0)

        return mask  # (1,512,512)

    def __getitem__(self, idx):

        file_name = self.file_names[idx]

        cloth_path = os.path.join(self.dataset_path, "cloth", file_name)
        image_path = os.path.join(self.dataset_path, "image", file_name)

        agnostic_path = os.path.join(self.dataset_path, "agnostic-v3.2", file_name)

        # ---- Load images ----
        cloth = self.load_rgb(cloth_path)
        image = self.load_rgb(image_path)
        agnostic = self.load_rgb(agnostic_path)


        concat_img = self.concat_images(cloth, image)
        concat_tensor = self.to_tensor(concat_img)



        mask_512 = self.extract_mask_from_agnostic(agnostic)

        mask_pad = self.zero_pad_left(T.ToPILImage()(mask_512))


        mask = (mask_pad > 0.5).float()

        return concat_tensor, mask

In [ ]:
trans = transforms.Compose([transforms.ToTensor()])

train_ds = VITONDataset(dataset_path=path+"/train")



In [ ]:
train_loader = DataLoader(train_ds,batch_size=32,shuffle=True,num_workers=8,
    pin_memory=True,
    persistent_workers=True)


In [ ]:
from google.colab import drive
import os


save_dir = "/content/drive/MyDrive/VTON_Project/checkpoints"
os.makedirs(save_dir, exist_ok=True)

In [ ]:
import torch.nn as nn
from diffusers import UNet2DConditionModel

unet = UNet2DConditionModel.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    subfolder="unet"
).to(device)#SD 1.5 INPAINTING MODELINI KULLANMAK GAYET MANTIKLI.


#unet parametrelerini freeze ediyoruz. sadece lora kullancaz
for param in unet.parameters():
    param.requires_grad = False


#Lora parameter efficient training (peft) tekniğidir.
#sadece attn1 katmanları yani self attn katmanlarına uyguluyorum
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=32,
    lora_alpha=32,

    # SADECE self-attn
    target_modules=[
        "attn1.to_q",
        "attn1.to_k",
        "attn1.to_v",
        "attn1.to_out.0"
    ],

    lora_dropout=0.1,
    bias="none"
)

unet = get_peft_model(unet, lora_config)



unet.print_trainable_parameters()

trainable = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print("Trainable params:", trainable)

print("LoRA sadece ATTN1'e uygulandı 🔥")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
An error occurred while trying to fetch stable-diffusion-v1-5/stable-diffusion-inpainting: stable-diffusion-v1-5/stable-diffusion-inpainting does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


trainable params: 3,194,880 || all params: 862,730,244 || trainable%: 0.3703
Trainable params: 3194880
LoRA sadece ATTN1'e uygulandı 🔥


In [ ]:
from diffusers.models.autoencoders import AutoencoderKL
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
vae = AutoencoderKL.from_pretrained("stable-diffusion-v1-5/stable-diffusion-inpainting", subfolder="vae").half().to(device)
vae.eval()

config.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

An error occurred while trying to fetch stable-diffusion-v1-5/stable-diffusion-inpainting: stable-diffusion-v1-5/stable-diffusion-inpainting does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


vae/diffusion_pytorch_model.bin:   0%|          | 0.00/335M [00:00<?, ?B/s]

AutoencoderKL(
  (encoder): Encoder(
    (conv_in): Conv2d(3, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (down_blocks): ModuleList(
      (0): DownEncoderBlock2D(
        (resnets): ModuleList(
          (0-1): 2 x ResnetBlock2D(
            (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
            (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm2): GroupNorm(32, 128, eps=1e-06, affine=True)
            (dropout): Dropout(p=0.0, inplace=False)
            (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (nonlinearity): SiLU()
          )
        )
        (downsamplers): ModuleList(
          (0): Downsample2D(
            (conv): Conv2d(128, 128, kernel_size=(3, 3), stride=(2, 2))
          )
        )
      )
      (1): DownEncoderBlock2D(
        (resnets): ModuleList(
          (0): ResnetBlock2D(
            (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
            (c

In [ ]:
noise_scheduler = DDPMScheduler.from_pretrained("stable-diffusion-v1-5/stable-diffusion-inpainting", subfolder="scheduler")

attn1 self attn
attn2 cross attn

input text embed = torch.zeros olcak. etkisini yok edicem.

üstüne lora yapıcam.

In [ ]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, unet.parameters()),
    lr=2e-4
)

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

lr_scheduler = CosineAnnealingLR(optimizer,T_max=30, eta_min=1e-5)

#BUNU SADECE BİR ŞEY LOAD EDECEKSEN ÇALIŞTIR SADECE

In [ ]:
ckpt_path = os.path.join(save_dir, "checkpoint_epoch_36.pt")

checkpoint = torch.load(ckpt_path, map_location=device)

unet.load_state_dict(checkpoint["unet"])
optimizer.load_state_dict(checkpoint["optimizer"])

start_epoch = checkpoint["epoch"] + 1
best_loss = checkpoint["loss"]

print(f"Checkpoint yüklendi. Epoch: {start_epoch}")
print("Loaded LR:", optimizer.param_groups[0]["lr"])

Checkpoint yüklendi. Epoch: 37
Loaded LR: 3.440124157964802e-05


In [ ]:
initial_lr = optimizer.param_groups[0]["lr"]
target_lr = 1e-4
decay_epochs = 3

In [ ]:
unet.enable_gradient_checkpointing()
unet = unet.to(memory_format=torch.channels_last)

torch.set_float32_matmul_precision("high")

In [ ]:
from tqdm import tqdm
import torch.nn.functional as F

scaler = torch.cuda.amp.GradScaler()

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

epochs = 100
best_loss = float("inf")

for epoch in range(start_epoch,epochs):

    unet.train()
    epoch_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for batch in pbar:


        images = batch[0].to(device)
        masks = batch[1].to(device)

        batch_size = images.shape[0]


        images = images * 2.0 - 1 #tensoru -1-1 arasına çekiyoruz SD VAE istediği aralık.
        masked_images = images * (1-masks)


        with torch.no_grad():
            latents = vae.encode(images).latent_dist.sample()
            latents = latents * vae.config.scaling_factor

            masked_latents = vae.encode(masked_images).latent_dist.sample()
            masked_latents = masked_latents * vae.config.scaling_factor


        masks = F.interpolate(masks,size = latents.shape[-2:],mode="nearest")#(B, C, H, W) dolayısıyla H,W



        noise = torch.randn_like(latents)#Noise sample ediyoruz gaussian distrubutiondan.
        timesteps = torch.randint(
            0,
            noise_scheduler.num_train_timesteps,
            (latents.shape[0],),
            device=device
        ).long()

        noisy_latents = noise_scheduler.add_noise(latents,noise,timesteps)

        model_input = torch.cat(tensors=
                                  [noisy_latents,
                                    masks,
                                    masked_latents],dim=1) #(B, C, H, W) dim 1 = Channel-wise concat

        encoder_hidden_states = torch.zeros(size = (latents.shape[0],77,768),device=device) # batchsize kadarlık bir zero embedding oluşturuyoruz.

        with torch.cuda.amp.autocast():

          noise_pred = unet(model_input,
                            timesteps,
                            encoder_hidden_states= encoder_hidden_states).sample

          loss = F.mse_loss(noise_pred,noise)



        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        pbar.set_postfix({"loss": loss.item()})


    avg_loss = epoch_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} Ortalama Loss: {avg_loss:.6f}")


    new_lr = target_lr

    for param_group in optimizer.param_groups:
        param_group["lr"] = new_lr

    print("Current LR:", optimizer.param_groups[0]["lr"])
    print("Current LR:", optimizer.param_groups[0]["lr"])


    if avg_loss < best_loss:
        best_loss = avg_loss

        torch.save(
            {
                "epoch": epoch,
                "unet": unet.state_dict(),
                "optimizer": optimizer.state_dict(),
                "loss": avg_loss,
            },
            os.path.join(save_dir, f"checkpoint_epoch_{epoch}.pt")
        )

/tmp/ipykernel_174920/3645101451.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 38/100: 100%|██████████| 364/364 [18:37<00:00,  3.07s/it, loss=0.0779]



Epoch 38 Ortalama Loss: 0.080373
Current LR: 0.0001
Current LR: 0.0001


Epoch 39/100: 100%|██████████| 364/364 [18:34<00:00,  3.06s/it, loss=0.123]



Epoch 39 Ortalama Loss: 0.079179
Current LR: 0.0001
Current LR: 0.0001


Epoch 40/100:  91%|█████████ | 332/364 [16:57<01:37,  3.05s/it, loss=0.0667]

In [ ]:
test_ds = VITONDataset(dataset_path=path+"/test")
test_loader = DataLoader(test_ds,batch_size = 1,shuffle=True)


In [ ]:
import torch
import torch.nn.functional as F
from torchvision.utils import save_image
from tqdm import tqdm

device = "cuda"

# -------------------------
# LOAD MODELS
# -------------------------
from diffusers import UNet2DConditionModel, AutoencoderKL, DDPMScheduler
from transformers import CLIPTokenizer, CLIPTextModel

unet = UNet2DConditionModel.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    subfolder="unet"
).to(device)

vae = AutoencoderKL.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    subfolder="vae"
).to(device)

scheduler = DDPMScheduler.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    subfolder="scheduler"
)

scheduler.set_timesteps(50)

tokenizer = CLIPTokenizer.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    subfolder="tokenizer"
)

text_encoder = CLIPTextModel.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    subfolder="text_encoder"
).to(device)

unet.eval()
vae.eval()
text_encoder.eval()

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# -------------------------
# INPUT
# -------------------------
batch = next(iter(test_loader))

image = batch[0].to(device) # [B,3,H,W]
mask  = batch[1].to(device)  # [B,1,128,64]

B = image.shape[0]

# -------------------------
# 1️⃣ MASK → IMAGE SIZE
# -------------------------
mask = F.interpolate(mask, size=image.shape[-2:], mode="nearest")
mask = (mask > 0.5).float()

# -------------------------
# 2️⃣ PIXEL SPACE MASKING
# -------------------------
masked_image = image * (1 - mask)

# -------------------------
# 3️⃣ VAE ENCODE
# -------------------------
with torch.no_grad():
    latents = vae.encode(image).latent_dist.sample() * 0.18215
    masked_latents = vae.encode(masked_image).latent_dist.sample() * 0.18215

# -------------------------
# 4️⃣ MASK → LATENT SIZE
# -------------------------
mask = F.interpolate(mask, size=latents.shape[-2:], mode="nearest")

# -------------------------
# 5️⃣ TEXT EMBEDDING
# -------------------------
prompt = ["a realistic clothing"] * B

inputs = tokenizer(
    prompt,
    padding="max_length",
    max_length=77,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    encoder_hidden_states = text_encoder(inputs.input_ids)[0]

# -------------------------
# 6️⃣ INIT NOISE
# -------------------------
noise = torch.randn_like(latents)

latents = latents * (1 - mask) + noise * mask

# -------------------------
# 7️⃣ SAMPLING LOOP
# -------------------------
for t in tqdm(scheduler.timesteps):

    t_batch = torch.full((B,), t, device=device, dtype=torch.long)

    model_input = torch.cat([
        latents,
        masked_latents,
        mask
    ], dim=1)

    with torch.no_grad():
        noise_pred = unet(
            model_input,
            t_batch,
            encoder_hidden_states=encoder_hidden_states
        ).sample

    latents = scheduler.step(noise_pred, t, latents).prev_sample

    # 🔒 keep non-masked area fixed
    latents = latents * mask + masked_latents * (1 - mask)

# -------------------------
# 8️⃣ DECODE
# -------------------------
with torch.no_grad():
    image = vae.decode(latents / 0.18215).sample

# [-1,1] → [0,1]
image = (image.clamp(-1, 1) + 1) / 2

# -------------------------
# 9️⃣ SAVE
# -------------------------
save_image(image, "inpainting_output.png")

print("✅ DONE")

An error occurred while trying to fetch stable-diffusion-v1-5/stable-diffusion-inpainting: stable-diffusion-v1-5/stable-diffusion-inpainting does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch stable-diffusion-v1-5/stable-diffusion-inpainting: stable-diffusion-v1-5/stable-diffusion-inpainting does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: stable-diffusion-v1-5/stable-diffusion-inpainting
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 50/50 [00:20<00:00,  2.40it/s]


✅ DONE
